In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import nrrd
import os
from model.LeNet5 import LeNet
from torch.utils.data import Dataset, DataLoader

In [14]:
seg_data,seg_header=nrrd.read(r'D:\Project\ACL\my_tool\Thuan dot 1\KHUAT THI THU HANG (22940170)\Segmentation.seg.nrrd')
data,header=nrrd.read(r'D:\Project\ACL\my_tool\Thuan dot 1\KHUAT THI THU HANG (22940170)\8 t2_tse_sag.nrrd')
print(f"Data shape: {data.shape}")
print(f"Segmentation shape: {seg_data.shape}") 
slice=data[:,:,1].shape
print(f"Slice shape: {slice}")

Data shape: (608, 608, 25)
Segmentation shape: (608, 608, 25)
Slice shape: (608, 608)


In [53]:
class MRI_data(Dataset):
    def __init__(self,root_nrrd_file_dir,root_nrrd_seg_dir,img_trasform=None,label_transform=None,target_size=(320,320)):
        data, _=nrrd.read(root_nrrd_file_dir)
        seg_data,_=nrrd.read(root_nrrd_seg_dir)
        self.images=data
        self.segs=seg_data
        self.img_transform=img_trasform
        self.label_transform=label_transform
        self.target_size=target_size
    def __len__(self):
        return self.images.shape[2]
    def __getitem__(self, index): 
        img=self.images[:,:,index].astype(np.uint8)
        label= 1 if self.segs[:,:,index].any() != 0 else 0
        if self.img_transform:
            img=self.img_transform(img)
        if self.label_transform:
            label=self.label_transform(label)
        return img,label 

In [55]:
img_transform=transforms.Compose([
    
    transforms.Resize((320,320)),
    transforms.ToTensor(),
])
label_transform=transforms.Compose([
    transforms.ToTensor(),
])

In [56]:
dataset=MRI_data(root_nrrd_file_dir=r'D:\Project\ACL\my_tool\Thuan dot 1\KHUAT THI THU HANG (22940170)\8 t2_tse_sag.nrrd',
                 root_nrrd_seg_dir=r'D:\Project\ACL\my_tool\Thuan dot 1\KHUAT THI THU HANG (22940170)\Segmentation.seg.nrrd',
                 img_trasform=img_transform,
                 label_transform=label_transform)
print(f"Dataset length: {len(dataset)}")
print(f"Sample image shape: {dataset[0][0].shape}, Sample label: {dataset[0][1]}")
test_loader=DataLoader(dataset=dataset,batch_size=4,shuffle=True)
for images,labels in test_loader:
    print(f"Batch image shape: {images.shape}, Batch labels: {labels}")
    break

Dataset length: 25


TypeError: Unexpected type <class 'numpy.ndarray'>

In [45]:
para = {
        'input_image_size': (320, 320),
        'input_channel': 1,
        'number_of_conv_layer':2,
        'number_of_fc_layer':2,
        'num_classes':2,
        'conv_channels': [6, 16],
        'fc_feature':[84,2]
    }
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=LeNet(para)
model.load_state_dict(torch.load('model.pth',weights_only=False))
model.to(device)
model.eval()

LeNet(
  (Conv_layer): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 6, kernel_size=(7, 7), stride=(1, 1), padding=(2, 2))
      (1): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(6, 16, kernel_size=(7, 7), stride=(1, 1), padding=(2, 2))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (FC_layer): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=97344, out_features=84, bias=True)
      (1): ReLU()
      (2): Linear(in_features=84, out_features=2, bias=True)
      (3): Softmax(dim=1)
    )
  )
)